# 01 URDF Combination

In [ ]:
from motionblender.app.utils import RobotInterface, MotionBlender
import os
import os.path as osp
import roma
import motionblender.lib.misc as misc
import torch.nn.functional as F
from loguru import logger
import torch.nn as nn
import torch.optim as optim
import numpy as np
import motionblender.lib.animate as anim
from jaxtyping import Float32
from tqdm.auto import tqdm, trange
import kinpy as kp
import torch
import transformations as tf
from torch import Tensor

def map_gripper_ctrl_to_joint_v(gv):
    v = max(min(gv, 100), 30)
    v = 0.8 * (v - 30) / 70
    return {
        'finger_1_joint_1': v,
        'finger_2_joint_1': v,
        'finger_middle_joint_1': v
    }

robot_joints = [
    'joint_0',
    'joint_1',
    'joint_2',
    'joint_3',
    'joint_4',
    'joint_5',
    'joint_6',
    'joint_7', 
    'ee',
    'palm',
    'left_finger_1',
    'right_finger_1',
    'left_finger_2',
    'right_finger_2',
]

robot_joints_from_origin = {
    'joint_0': 'iiwa_link_0',
    'joint_1': 'iiwa_link_1',
    'joint_2': 'iiwa_link_2',
    'joint_3': 'iiwa_link_3',
    'joint_4': 'iiwa_link_4',
    'joint_5': 'iiwa_link_5',
    'joint_6': 'iiwa_link_6',
    'joint_7': 'iiwa_link_7',
    'ee': 'iiwa_link_ee',
    'palm': 'palm',
    'left_finger_1': ['finger_middle_link_0'],
    'right_finger_1': ['finger_1_link_0', 'finger_2_link_0'],
    'left_finger_2': ['finger_middle_link_3'],
    'right_finger_2': ['finger_1_link_3', 'finger_2_link_3'],
}

robot_connections = [
    ['joint_0', 'joint_1'],
    ['joint_1', 'joint_2'],
    ['joint_2', 'joint_3'],
    ['joint_3', 'joint_4'],
    ['joint_4', 'joint_5'],
    ['joint_5', 'joint_6'],
    ['joint_6', 'joint_7'],
    ['joint_7', 'ee'],
    ['ee', 'palm'],
    ['palm', 'left_finger_1'],
    ['palm', 'right_finger_1'],
    ['left_finger_1', 'left_finger_2'],
    ['right_finger_1', 'right_finger_2'],
]

robot_connections_int = [(robot_joints.index(a), robot_joints.index(b)) for a, b in robot_connections]

default_joint_values = {'iiwa_joint_1': -0.15783709287643433,
 'iiwa_joint_2': 0.48583802580833435,
 'iiwa_joint_3': 1.0546152225288097e-05,
 'iiwa_joint_4': -1.6795016527175903,
 'iiwa_joint_5': 0.9391155242919922,
 'iiwa_joint_6': 1.027316689491272,
 'iiwa_joint_7': -1.273979663848877,
 'palm_finger_1_joint': -0.16,
 'palm_finger_2_joint': 0.16,
 'finger_middle_joint_3': 0,
 'finger_middle_joint_2': 0,
 'finger_2_joint_3': 0,
 'finger_2_joint_2': 0,
 'finger_1_joint_3': 0,
 'finger_1_joint_2': 0,
 'finger_1_joint_1': 0.0,
 'finger_2_joint_1': 0.0,
 'finger_middle_joint_1': 0.0}

class RobotInterface:
    def __init__(self):
        self.buf = {
            'degree': 50,
            'pose': None,
            'stale': True,
            'rot6d': None
        }
        self.inited = False

    def initialize(self, motion_module: MotionBlender) -> None:
        raise NotImplementedError
    
    def get_joint_rotations(self) -> Float32[Tensor, "j 6"]:
        raise NotImplementedError

    def set_gripper_degree(self, degree: int) -> None:
        self.buf['degree'] = degree
        self.buf['stale'] = True

    def set_ee_pose(self, ee_pose: Float32[Tensor, "4 4"]) -> None:
        self.buf['pose'] = ee_pose
        self.buf['stale'] = True

    def get_ee_pose(self) -> Float32[Tensor, "4 4"]:
        return self.buf['pose']


class Kuka(RobotInterface):
    def __init__(self, iiwa_path=os.path.dirname(__file__) + "/iiwa/kuka.urdf"):
        super().__init__()
        txt = open(iiwa_path).read()
        self.iiwa_serial_chain = kp.build_serial_chain_from_urdf(txt, 'palm')
        self.iiwa_chain = kp.build_chain_from_urdf(txt)
        self.joint_names = [f'iiwa_joint_{i}' for i in range(1, 8)]
        self.curr_jvs = np.array([default_joint_values[jname] for jname in self.joint_names])
        self.buf['rot6d'] = None
    
    def _joint_values_to_rot6d(self, joint_values):
        link_poses = self.iiwa_chain.forward_kinematics(joint_values)
        robot_joints_values = {}
        for rj_name, parents in robot_joints_from_origin.items(): 
            if isinstance(parents, list):
                values = []
                for p in parents:
                    values.append(link_poses[p].pos)
                robot_joints_values[rj_name] = sum(values) / len(values)
            else:
                robot_joints_values[rj_name] = link_poses[parents].pos

        joint_positions = []
        for rj_name in robot_joints:
            joint_positions.append(robot_joints_values[rj_name])
        joint_positions = torch.from_numpy(np.array(joint_positions)).float()
        anim_chain = anim.inverse_kinematic(joint_positions, robot_connections_int)
        rot6d = anim.retrieve_tensor_from_chain(anim_chain, 'rot6d')
        return rot6d

    def initialize(self, motion_module: MotionBlender) -> None:
        gripper_poses = getattr(motion_module, 'gripper_poses', [])
        if len(gripper_poses) == 0:
            if hasattr(motion_module, 'original_cano_t'):
                original_cano_t = motion_module.original_cano_t
            else:
                original_cano_t = 323

            ROBOT_ROOT_DATASET_DIR = './datasets/robot/'
            img_id = misc.load_json(osp.join(ROBOT_ROOT_DATASET_DIR, 'dataset.json'))['ids'][original_cano_t]
            jv = misc.load_cpkl(osp.join(ROBOT_ROOT_DATASET_DIR, 'robot_rawdata.pkl'))['joint_pos_list'][int(img_id)]
            self.curr_jvs = np.array(jv)
            end_pose_tf = self.iiwa_serial_chain.forward_kinematics(self.curr_jvs)
            ee_pose_robot_base = torch.from_numpy(end_pose_tf.matrix()).float().to(next(motion_module.parameters()).device)
            self.buf['pose'] = ee_pose_robot_base
            self.buf['degree'] = 50.
        else:
            self.buf['pose'] = gripper_poses[0]
            self.buf['degree'] = motion_module.gripper_degrees[0]

        self.get_joint_rotations()
        self.inited = True
    
    def get_joint_rotations(self, refresh=False) -> Float32[Tensor, "j 6"]:
        if not self.buf['stale'] and self.buf['rot6d'] is not None and not refresh:
            return self.buf['rot6d']

        matrix = self.buf['pose'].cpu().numpy()
        trans = kp.Transform(rot=tf.quaternion_from_matrix(matrix), pos=matrix[:3, 3])
        self.curr_jvs = self.iiwa_serial_chain.inverse_kinematics(trans, self.curr_jvs)
        joint_values = {**default_joint_values, **map_gripper_ctrl_to_joint_v(self.buf['degree']), 
                        **dict(zip(self.joint_names, self.curr_jvs))}
        rot6d = self._joint_values_to_rot6d(joint_values)
        self.buf['rot6d'] = rot6d.to(self.buf['pose'].device)
        self.buf['stale'] = False
        return self.buf['rot6d']

robot = Kuka()


gs_modules, motion_modules, _, gaussian_names = misc.load_cpkl("outputs/mb/robot/okish/toy/ckpt.robot.cpkl")
robot.initialize(motion_modules['robot'])
robot.buf['pose'][:3, 3] += (torch.rand(3).cuda() * 0.3)
robot.buf['pose'][:3, :3] = roma.random_rotmat().cuda() @ robot.buf['pose'][:3, :3]
print(robot.get_joint_rotations(refresh=True))

In [9]:
import kinpy as kp

urdf_path = 'my_data/ur_description/urdf/ur5_robot.urdf'
# urdf_path = 'my_data/DH13_right_palm_URDF/urdf/DH13_right_palm_URDF.urdf'
# urdf_path = 'my_data/ur5_dh13_combined.urdf'

txt = open(urdf_path, 'rb').read()
chain = kp.build_chain_from_urdf(txt)
print(chain)
# serial_chain = kp.build_serial_chain_from_urdf(txt, 'ee_link')
# print(serial_chain)



world_frame
└──── base_link_frame
      ├──── shoulder_link_frame
      │     └──── upper_arm_link_frame
      │           └──── forearm_link_frame
      │                 └──── wrist_1_link_frame
      │                       └──── wrist_2_link_frame
      │                             └──── wrist_3_link_frame
      │                                   ├──── ee_link_frame
      │                                   │     └──── connection_bracket_link_frame
      │                                   │           └──── right_palm_link_frame
      │                                   └──── tool0_frame
      └──── base_frame



In [2]:
import cloudpickle as cpickle

def load_cpkl(path):
    with open(path, 'rb') as f:
        return cpickle.load(f)
    
pkl_path = 'datasets/robot/robot/robot_rawdata.pkl'
data = load_cpkl(pkl_path)
data


{'save_freq': 0.5,
 'num_steps': 30591,
 'cam_K': array([[616.99243,   0.     , 314.10406],
        [  0.     , 616.9546 , 236.41193],
        [  0.     ,   0.     ,   1.     ]], dtype=float32),
 'eef_pose_list': [array([ 0.5592351 ,  0.01731292,  0.3996791 , -0.26527479,  0.88532478,
          0.30255493,  0.23300174]),
  array([ 0.55923513,  0.01731272,  0.3996791 , -0.26527456,  0.8853249 ,
          0.30255483,  0.23300175]),
  array([ 0.5592351 ,  0.01731298,  0.3996791 , -0.26527468,  0.88532484,
          0.30255498,  0.23300167]),
  array([ 0.55923514,  0.01731281,  0.39967918, -0.2652746 ,  0.88532484,
          0.30255482,  0.23300184]),
  array([ 0.55923518,  0.01731297,  0.39967916, -0.26527472,  0.88532484,
          0.30255485,  0.23300178]),
  array([ 0.55923515,  0.01731286,  0.39967913, -0.26527463,  0.88532484,
          0.30255496,  0.23300171]),
  array([ 0.55923513,  0.01731286,  0.39967908, -0.26527463,  0.88532484,
          0.30255489,  0.23300171]),
  array([ 0

# 02 SAPIEN

In [3]:
import sapien as sapien
from sapien.utils import Viewer
import numpy as np


def main():
    scene = sapien.Scene()  # Create an instance of simulation world (aka scene)
    scene.set_timestep(1 / 100.0)  # Set the simulation frequency

    # NOTE: How to build (rigid bodies) is elaborated in create_actors.py
    scene.add_ground(altitude=0)  # Add a ground
    actor_builder = scene.create_actor_builder()
    actor_builder.add_box_collision(half_size=[0.5, 0.5, 0.5])
    actor_builder.add_box_visual(half_size=[0.5, 0.5, 0.5], material=[1.0, 0.0, 0.0])
    box = actor_builder.build(name="box")  # Add a box
    box.set_pose(sapien.Pose(p=[0, 0, 0.5]))

    # Add some lights so that you can observe the scene
    scene.set_ambient_light([0.5, 0.5, 0.5])
    scene.add_directional_light([0, 1, -1], [0.5, 0.5, 0.5])

    viewer = scene.create_viewer()  # Create a viewer (window)

    # The coordinate frame in Sapien is: x(forward), y(left), z(upward)
    # The principle axis of the camera is the x-axis
    viewer.set_camera_xyz(x=-4, y=0, z=2)
    # The rotation of the free camera is represented as [roll(x), pitch(-y), yaw(-z)]
    # The camera now looks at the origin
    viewer.set_camera_rpy(r=0, p=-np.arctan2(2, 4), y=0)
    viewer.window.set_camera_parameters(near=0.05, far=100, fovy=1)

    while not viewer.closed:  # Press key q to quit
        scene.step()  # Simulate the world
        scene.update_render()  # Update the world to the renderer
        viewer.render()


if __name__ == "__main__":
    main()

In [12]:
import sapien


def demo(fix_root_link, balance_passive_force):
    scene = sapien.Scene()
    scene.add_ground(0)

    scene.set_ambient_light([0.5, 0.5, 0.5])
    scene.add_directional_light([0, 1, -1], [0.5, 0.5, 0.5])

    viewer = scene.create_viewer()
    viewer.set_camera_xyz(x=-2, y=0, z=1)
    viewer.set_camera_rpy(r=0, p=-0.3, y=0)

    # Load URDF
    loader = scene.create_urdf_loader()
    loader.fix_root_link = fix_root_link
    # robot: sapien.Articulation = loader.load("./my_data/DH13_right_palm_URDF/urdf/DH13_right_palm_URDF.urdf")
    # robot: sapien.Articulation = loader.load("./my_data/ur_description/urdf/ur5_robot.urdf")
    robot: sapien.Articulation = loader.load("./my_data/ur5_dh13_combined.urdf")
    robot.set_root_pose(sapien.Pose([0, 0, 0], [1, 0, 0, 0]))

    # # Set initial joint positions
    # arm_init_qpos = [4.71, 2.84, 0, 0.75, 4.62, 4.48, 4.88]
    # gripper_init_qpos = [0, 0, 0, 0, 0, 0]
    # init_qpos = arm_init_qpos + gripper_init_qpos
    # robot.set_qpos(init_qpos)

    while not viewer.closed:
        for _ in range(4):  # render every 4 steps
            if balance_passive_force:
                qf = robot.compute_passive_force(
                    gravity=True,
                    coriolis_and_centrifugal=True,
                )
                robot.set_qf(qf)
            scene.step()
        scene.update_render()
        viewer.render()


demo(fix_root_link=True, balance_passive_force=True)

In [ ]:
"""
  <!-- Connection Bracket Link -->
  <link name="connection_bracket_link">
    <visual>
      <geometry>
        <!-- 进一步缩小连接件，并调整其局部位置 -->
        <mesh filename="package://connector/UR5_DH13_tutai.STL" scale="0.0004 0.0004 0.0004"/>
      </geometry>
      <origin xyz="0.0 0.0 0.0" rpy="0.0 0.0 0.0"/>
    </visual>
    <collision>
      <geometry>
        <mesh filename="package://connector/UR5_DH13_tutai.STL" scale="0.0004 0.0004 0.0004"/>
      </geometry>
      <origin xyz="0.0 0.0 0.0" rpy="0.0 0.0 0.0"/>
    </collision>
    <inertial>
      <mass value="0.05"/>
      <origin xyz="0.0 0.0 0.0"/>
      <inertia ixx="0.0005" ixy="0.0" ixz="0.0" iyy="0.0005" iyz="0.0" izz="0.0005"/>
    </inertial>
  </link>

  <!-- Connect UR5 end-effector to connection bracket -->
  <joint name="ur5_to_bracket" type="fixed">
    <parent link="ee_link"/>
    <child link="connection_bracket_link"/>
    <origin xyz="0.03 -0.02 0.0" rpy="0.0 0.0 1.5708"/>
  </joint>

    <!-- Connect connection bracket to DH13 hand -->
  <joint name="bracket_to_dh13" type="fixed">
    <parent link="connection_bracket_link"/>
    <child link="right_palm_link"/>
    <!-- 调整DH13手部相对于连接件的位置和方向 -->
    <origin xyz="0.0 0.0 0.015" rpy="0.0 0.0 -1.5708"/>
  </joint>

"""

# 03 Data Process

In [1]:
import cv2
import os

def images_to_video(image_folder, video_name, fps=30):
    images = [img for img in os.listdir(image_folder) if img.endswith(".jpg") or img.endswith(".png")]
    images.sort()  # Sort images by name

    if not images:
        print("No images found in the specified folder.")
        return

    first_image = cv2.imread(os.path.join(image_folder, images[0]))
    height, width, layers = first_image.shape

    video = cv2.VideoWriter(video_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    for image in images:
        video.write(cv2.imread(os.path.join(image_folder, image)))

    video.release()
    print(f"Video {video_name} created successfully.")

if __name__ == "__main__":
    image_folder = 'datasets/robot/robot/rgb'  # Path to the images folder
    video_name = 'output_video.mp4'  # Output video file name
    images_to_video(image_folder, video_name)

Video output_video.mp4 created successfully.
